In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.functions import date_format
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime

In [0]:
def ler_ultima_particao_tabela_spark(spark, source_table):
  """
  Essa função ler a ultima partição das Tabelas no formato delta baaseado na coluna de data_processamento
  """
  try: 
    # Mais performatica para pegar os metadados
    show_partitions_df = spark.sql(f"SHOW PARTITIONS {source_table}")
    # maior partição da data_processamento
    max_partition = show_partitions_df.agg(f.max("data_processamento")).collect()[0][0]
    print(f"partição maxima {max_partition}")

    # pegar o dataframe com maior partição
    return spark.table(f"{source_table}")\
                      .filter(f.col("data_processamento") == max_partition)
  except Exception as e:
    print(f"Erro ao ler o caminho {source_table}: {e}")
    return None


### 1. gold_fato_diario

In [0]:
gold_fato_diario  = ler_ultima_particao_tabela_spark(spark,  "workspace.case_spark_cvm.gold_fato_diario")

In [0]:
display(gold_fato_diario)

### 2. Pegar Ultima Data do Drawdown

In [0]:
window_drawdown = Window.partitionBy("cnpj_fundo_classe").orderBy(f.col("drawdown_maximo_historico").asc())

df_drawdown = gold_fato_diario\
    .withColumn("rn", f.row_number().over(window_drawdown))\
    .filter(f.col("rn") ==  1)\
    .select("cnpj_fundo_classe", f.col("dt_comptc").alias("data_drawdown_maximo"))

In [0]:
gold_fato_diario = gold_fato_diario\
    .join(
        df_drawdown,
        'cnpj_fundo_classe',
        "left"
    )

### 4. Pegando a Ultima Data do Fundo

In [0]:
window_ultima_data = Window.partitionBy("cnpj_fundo_classe")
window_drawdown = Window.partitionBy("cnpj_fundo_classe").orderBy(f.col("drawdown_maximo_historico").asc())

gold_fato_diario = gold_fato_diario\
    .withColumn("ultima_dt_fundo", f.max(f.col("dt_comptc")).over(window_ultima_data))

### 5. Selecionando as Colunas de Risco

In [0]:
gold_cubo_risco = gold_fato_diario\
    .filter(f.col("dt_comptc") == f.col("ultima_dt_fundo"))\
    .withColumn(
        "classificacao_risco",
        f.when(f.col("volatilidade_252d") < 0.02, "BAIXO")
         .when((f.col("volatilidade_252d") >= 0.02) & (f.col("volatilidade_252d") <= 0.10), "MÉDIO")
         .otherwise("ALTO")
    )\
    .select(
        "cnpj_fundo_classe",
        f.col("dt_comptc").alias("dt_referencia"),
        "volatilidade_21d",
        "volatilidade_63d",
        "volatilidade_252d",
        "drawdown_maximo_252d",
        "drawdown_maximo_historico",
        "data_drawdown_maximo",
        "var_95_252d",
        "classificacao_risco"
    )

In [0]:
gold_cubo_risco = gold_cubo_risco.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

data_proc = int(datetime.now().strftime(f"%Y%m%d"))

gold_cubo_risco.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.gold_cubo_risco")

In [0]:
display(gold_cubo_risco)